In [1]:
import pandas as pd
import numpy as np

from scipy.stats import wasserstein_distance

In [13]:
REFERENCE_CSV = (
    "../../data/processed/CTB/sampled_real_eventlogs/s6_sample_24.000_eventlog_target_rank_features.csv"
)

SIM_CSV = (
    "../../data/processed/CTB/prosit_simulations/sim_log_s6_sample_24.000_depth10_maxConc_2.csv"
)

ref = pd.read_csv(REFERENCE_CSV)
sim = pd.read_csv(SIM_CSV)

print(
    f"Reference events: {len(ref):,}"
)

print(
    f"Simulation events: {len(sim):,}"
)

Reference events: 74,066
Simulation events: 74,988


In [5]:
# =============CALCULATE SIM KPIS=============================================
# CALCULATE SIMULATION KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    sim[col] = pd.to_datetime(sim[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

sim["waiting_time"] = (
    sim["start:timestamp"]
    - sim["enabled:timestamp"]
).dt.total_seconds() / 60

sim["service_time"] = (
    sim["time:timestamp"]
    - sim["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    sim.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    sim.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

sim["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

sim_rmg = sim[
    sim["concept:name"]
    .isin(rmg_activities)
].copy()

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Simulation RMG events: {len(sim_rmg):,}"
)

Reference RMG events: 24,000
Simulation RMG events: 25,205


In [6]:
#========== CALCULTATE REF KPIS =================
# ==========================================================
# CALCULATE REFERENCE KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    ref[col] = pd.to_datetime(ref[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

ref["waiting_time"] = (
    ref["start:timestamp"]
    - ref["enabled:timestamp"]
).dt.total_seconds() / 60

ref["service_time"] = (
    ref["time:timestamp"]
    - ref["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

ref_case_start = (
    ref.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

ref_case_end = (
    ref.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

ref["turnaround_time"] = (
    ref_case_end
    - ref_case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Reference cases: "
    f"{ref['case:concept:name'].nunique():,}"
)

print(
    "\nKPI columns created:"
)

print(
    [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]
)

Reference RMG events: 24,000
Reference cases: 24,000

KPI columns created:
['waiting_time', 'service_time', 'turnaround_time']


In [7]:
# ==========================================================
# WAITING TIME BY ACTIVITY
# ==========================================================

activity_waiting = pd.DataFrame({
    "ref_mean_waiting":
        ref.groupby(
            "concept:name"
        )["waiting_time"]
        .mean(),

    "sim_mean_waiting":
        sim.groupby(
            "concept:name"
        )["waiting_time"]
        .mean()
})

activity_waiting["difference"] = (
    activity_waiting["sim_mean_waiting"]
    - activity_waiting["ref_mean_waiting"]
)

activity_waiting.sort_values(
    "difference",
    ascending=False
)

,ref_mean_waiting,sim_mean_waiting,difference
concept:name,,,
HO2_delivery,13.656250,102.615077,88.958827
HO2_mixed,7.476923,90.526398,83.049475
HO2_receive,13.007843,68.546443,55.538600
RMG_delivery,6.166911,27.857425,21.690515
RMG_receive,7.623269,27.804862,20.181592
RMG_mixed,7.989294,24.179326,16.190032
Gate Out,8.360875,13.628070,5.267195
LL_receive,3.289796,8.000000,4.710204
LL_mixed,2.581283,6.592553,4.011269


In [8]:
# ==========================================================
# WAITING TIME BY RESOURCE
# ==========================================================

resource_waiting = pd.DataFrame({

    "ref_mean_waiting":
        ref.groupby(
            "org:resource"
        )["waiting_time"]
        .mean(),

    "sim_mean_waiting":
        sim.groupby(
            "org:resource"
        )["waiting_time"]
        .mean()

})

resource_waiting["difference"] = (
    resource_waiting["sim_mean_waiting"]
    - resource_waiting["ref_mean_waiting"]
)

resource_waiting.sort_values(
    "difference",
    ascending=False
)

,ref_mean_waiting,sim_mean_waiting,difference
org:resource,,,
HO2,12.560000,93.861394,81.301394
T08,10.349943,72.622519,62.272576
T22,4.836626,49.548713,44.712088
T07,7.733112,50.926494,43.193382
T18,7.688843,38.342523,30.653680
T25,6.324124,36.226858,29.902734
T09,7.406348,36.484515,29.078167
T11,8.709797,36.113467,27.403671
T26,7.694764,34.268288,26.573524


In [9]:
# ==========================================================
# RESOURCE FREQUENCY
# ==========================================================

ref_res = (
    ref["org:resource"]
    .value_counts()
    .rename("reference")
)

sim_res = (
    sim["org:resource"]
    .value_counts()
    .rename("simulation")
)

resource_load = pd.concat(
    [ref_res, sim_res],
    axis=1
).fillna(0)

resource_load["difference"] = (
    resource_load["simulation"]
    - resource_load["reference"]
)

resource_load.sort_values(
    "difference",
    ascending=False
)

,reference,simulation,difference
HO2,250,1525,1275
T15,932,1311,379
T06,1081,1378,297
T18,977,1166,189
T12,1183,1341,158
T26,783,937,154
T14,1412,1541,129
T17,1257,1381,124
T13,1529,1647,118
T24,1292,1398,106


In [10]:
# ==========================================================
# TOTAL WAITING CONTRIBUTION
# ==========================================================

ref_total_waiting = (
    ref.groupby(
        "org:resource"
    )["waiting_time"]
    .sum()
)

sim_total_waiting = (
    sim.groupby(
        "org:resource"
    )["waiting_time"]
    .sum()
)

waiting_contribution = pd.DataFrame({

    "ref_waiting":
        ref_total_waiting,

    "sim_waiting":
        sim_total_waiting

})

waiting_contribution["extra_waiting"] = (
    waiting_contribution["sim_waiting"]
    - waiting_contribution["ref_waiting"]
)

waiting_contribution.sort_values(
    "extra_waiting",
    ascending=False
)

,ref_waiting,sim_waiting,extra_waiting
org:resource,,,
HO2,3140.0,143138.625316,139998.625316
Res.GateOut,200661.0,327073.669753,126412.669753
T08,9139.0,59042.108026,49903.108026
T22,5533.1,47665.862346,42132.762346
T18,7512.0,44707.382100,37195.382100
T07,6983.0,40792.121606,33809.121606
T11,9424.0,40699.877779,31275.877779
T21,5752.0,35453.545680,29701.545680
T06,9614.0,39197.643212,29583.643212


In [11]:
waiting_contribution = pd.DataFrame({

    "ref_total":
    ref.groupby("org:resource")
       ["waiting_time"].sum(),

    "sim_total":
    sim.groupby("org:resource")
       ["waiting_time"].sum()
})

waiting_contribution["extra_waiting"] = (
    waiting_contribution["sim_total"]
    - waiting_contribution["ref_total"]
)

waiting_contribution.sort_values(
    "extra_waiting",
    ascending=False
).head(10)

,ref_total,sim_total,extra_waiting
org:resource,,,
HO2,3140.0,143138.625316,139998.625316
Res.GateOut,200661.0,327073.669753,126412.669753
T08,9139.0,59042.108026,49903.108026
T22,5533.1,47665.862346,42132.762346
T18,7512.0,44707.382100,37195.382100
T07,6983.0,40792.121606,33809.121606
T11,9424.0,40699.877779,31275.877779
T21,5752.0,35453.545680,29701.545680
T06,9614.0,39197.643212,29583.643212
